# Task 3 training
Original defaults are retained. For the five revised runs, use `train_recovery_v1.ipynb`.


In [ ]:
from pathlib import Path
import json
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if p.name == 'Task 3'
             and (p / 'task3/configs/erm.yaml').is_file()), None)
if REPO is None:
    raise RuntimeError('Open Jupyter inside Task 3.')
ROOT = REPO.parent
TASK2 = ROOT / 'Task 2'
DATA_ROOT = ROOT / 'data/PACS'
SPLIT_PATH = TASK2 / 'shared/splits/pacs_sketch_seed6304.json'
BASELINE = TASK2 / 'task2/results/source_only/best.pt'
RESULTS = REPO / 'task3/results'
CONFIG = json.loads((REPO / 'task3/configs/erm.yaml').read_text())

def load_notebook(path):
    path = Path(path)
    document = json.loads(path.read_text(encoding='utf-8'))
    for index, cell in enumerate(document['cells']):
        if cell['cell_type'] == 'code':
            source = cell['source']
            source = ''.join(source) if isinstance(source, list) else source
            exec(compile(source, f'{path}:cell-{index}', 'exec'), globals())

# Only definitions are loaded here; no target dataset is constructed.
load_notebook(TASK2 / 'shared/pacs_protocol.ipynb')
load_notebook(TASK2 / 'shared/pacs.ipynb')
load_notebook(REPO / 'task3/models/classifier_head.ipynb')
load_notebook(REPO / 'task3/models/backbone.ipynb')
load_notebook(TASK2 / 'task2/methods/dan.ipynb')  # Exact shared MMD implementation.

import random
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

load_notebook(REPO / 'task3/selection/source_validation.ipynb')
load_notebook(REPO / 'task3/methods/erm.ipynb')
load_notebook(REPO / 'task3/methods/dan_dg.ipynb')
load_notebook(REPO / 'task3/methods/sam.ipynb')
load_notebook(REPO / 'task3/evaluation/source_domain_separability.ipynb')
load_notebook(REPO / 'task3/evaluation/sharpness.ipynb')
load_notebook(REPO / 'task3/evaluation/domain_metrics.ipynb')


In [ ]:
"""Common PACS training loop. Source-only mode never loads target data."""
import json
import math
import random
import platform
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.auto import tqdm


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class DomainStream:
    """Cycle shuffled loaders, retaining tails so every update has exactly eight/domain."""
    def __init__(self, loader):
        self.loader = loader
        self.iterator = iter(loader)
        self.pending = None

    def take(self, count):
        xs, ys = [], []
        remaining = count
        while remaining:
            if self.pending is None:
                try:
                    self.pending = next(self.iterator)
                except StopIteration:
                    self.iterator = iter(self.loader)
                    self.pending = next(self.iterator)
            x, y = self.pending
            n = min(remaining, len(y))
            xs.append(x[:n]); ys.append(y[:n])
            self.pending = (x[n:], y[n:]) if len(y) > n else None
            remaining -= n
        return torch.cat(xs), torch.cat(ys)


def train_pacs(data_root, split_path, output_dir, config=None, method='source_only'):
    if method not in ('dan_dg', 'sam'):
        raise ValueError(f'Unsupported method: {method}')
    cfg = dict(CONFIG if config is None else config)
    validate_config(cfg, method)
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    checkpoint = out / 'best.pt'
    if checkpoint.exists():
        raise FileExistsError(f'Preserving existing baseline: {checkpoint}. Load it or choose a new output directory.')
    seed_everything(cfg['seed'])
    splits = prepare_splits(data_root, split_path)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loaders, val_loaders = {}, {}
    for i, domain in enumerate(SOURCES):
        train_loaders[domain] = DataLoader(
            PACSSource(data_root, splits['domains'][domain]['train'], True),
            batch_size=cfg['batch_per_domain'], shuffle=True, num_workers=cfg['workers'],
            generator=torch.Generator().manual_seed(cfg['seed'] + i), drop_last=False)
        val_loaders[domain] = DataLoader(
            PACSSource(data_root, splits['domains'][domain]['val']),
            batch_size=64, shuffle=False, num_workers=cfg['workers'])
    torch.hub.set_dir(str(Path(data_root).parent / 'torch_cache'))
    initial_path = cfg.get('initial_checkpoint')
    model = make_model(pretrained=not bool(initial_path)).to(device)
    if initial_path:
        initial = torch.load(initial_path, map_location='cpu', weights_only=True)
        if initial.get('method') != 'source_only' or initial['split_sha256'] != split_hash(split_path):
            raise ValueError('Recovery requires the matching source-only checkpoint.')
        if initial['class_to_idx'] != splits['class_to_idx']:
            raise ValueError('Initialization class mapping mismatch.')
        model.load_state_dict(initial['model_state'])
        cfg['initial_checkpoint_sha256'] = hashlib.sha256(Path(initial_path).read_bytes()).hexdigest()
        del initial
    if cfg.get('freeze_batchnorm_affine', False):
        for layer in model.modules():
            if isinstance(layer, nn.modules.batchnorm._BatchNorm):
                layer.requires_grad_(False)
    parameters = list(model.parameters())
    if 'backbone_learning_rate' in cfg:
        head_ids = {id(p) for p in model.fc.parameters()}
        groups = [dict(params=[p for p in model.parameters() if p.requires_grad and id(p) not in head_ids],
                       lr=cfg['backbone_learning_rate']),
                  dict(params=list(model.fc.parameters()), lr=cfg['learning_rate'])]
        optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    else:
        optimizer = torch.optim.AdamW(parameters, lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])
    metadata = dict(config=cfg, split_sha256=split_hash(split_path), class_to_idx=splits['class_to_idx'],
                    environment=dict(python=platform.python_version(), torch=str(torch.__version__),
                    torchvision=str(torchvision.__version__), device=str(device),
                    gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else None))
    metadata['method'] = method
    (out / 'config.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    steps = math.ceil(max(len(loader.dataset) for loader in train_loaders.values()) / cfg['batch_per_domain'])
    history, best, stale = [], -float('inf'), 0
    for epoch in range(1, cfg['epochs'] + 1):
        training_mode(model)
        streams = {d: DomainStream(loader) for d, loader in train_loaders.items()}
        total_loss, correct, count = 0.0, 0, 0
        total_classification, total_alignment = 0.0, 0.0
        total_perturbed = 0.0
        weighted_total, ramp_total, gradient_total = 0.0, 0.0, 0.0
        for step in tqdm(range(steps), desc=f'Epoch {epoch}/{cfg["epochs"]}', leave=False):
            batches = [streams[d].take(cfg['batch_per_domain']) for d in SOURCES]
            x = torch.cat([b[0] for b in batches]).to(device)
            y = torch.cat([b[1] for b in batches]).to(device)
            optimizer.zero_grad(set_to_none=True)
            ramp_epochs = cfg.get('alignment_ramp_epochs', 0)
            ramp = min(1., ((epoch - 1)*steps + step + 1)/(ramp_epochs*steps)) if ramp_epochs else 1.
            ramp_total += ramp
            step_cfg = dict(cfg)
            for key in ('lambda_mmd', 'lambda_dg', 'max_grl_strength'):
                if key in step_cfg:
                    step_cfg[key] *= ramp
            if method == 'sam':
                logits, loss, perturbed_loss = sam_step(model, optimizer, x, y, cfg['rho'])
                total_perturbed += perturbed_loss.item() * len(y)
            else:
                logits, loss, classification, alignment = dan_dg_objective(model, x, y, step_cfg)
                total_classification += classification.item() * len(y)
                total_alignment += alignment.item() * len(y)
                weighted_total += (loss - classification).item() * len(y)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite training loss')
            if method != 'sam':
                loss.backward()
                if cfg.get('gradient_clip_norm') is not None:
                    gradient_total += float(torch.nn.utils.clip_grad_norm_(parameters, cfg['gradient_clip_norm'], error_if_nonfinite=True))
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item(); count += len(y)
        metrics, _ = evaluate(model, val_loaders, device)
        score = float(metrics.macro_f1.mean())
        row = dict(epoch=epoch, train_loss=total_loss/count, train_accuracy=correct/count,
                   mean_val_macro_f1=score, mean_val_accuracy=float(metrics.accuracy.mean()),
                   mean_val_loss=float(metrics.loss.mean()))
        if method == 'sam':
            row.update(classification_loss=total_loss/count,
                       perturbed_classification_loss=total_perturbed/count)
        elif method == 'dan_dg':
            row.update(classification_loss=total_classification/count,
                       mmd_loss=total_alignment/count,
                       weighted_mmd_loss=weighted_total/count)
        if cfg.get('alignment_ramp_epochs', 0):
            row['mean_alignment_ramp'] = ramp_total / steps
        if cfg.get('gradient_clip_norm') is not None:
            row['mean_gradient_norm_before_clip'] = gradient_total / steps
        for item in metrics.to_dict('records'):
            for key in ('accuracy', 'macro_f1', 'loss'):
                row[f'{item["domain"]}_{key}'] = item[key]
        history.append(row)
        pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
        eligible = epoch >= cfg.get('selection_start_epoch', 1)
        if eligible and score > best:
            best, stale = score, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            payload = dict(model_state=state, epoch=epoch, best_source_macro_f1=best, **metadata)
            torch.save(payload, checkpoint)
        elif eligible:
            stale += 1
        print(f'Epoch {epoch}: loss={row["train_loss"]:.4f}, mean source macro-F1={score:.4f}, patience={stale}/{cfg["patience"]}', flush=True)
        if stale >= cfg['patience']:
            break
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    model.load_state_dict(saved['model_state'])
    metrics, predictions = evaluate(model, val_loaders, device)
    metrics.to_csv(out / 'source_validation.csv', index=False)
    predictions.to_csv(out / 'source_validation_predictions.csv', index=False)
    (out / 'completion.json').write_text(json.dumps(dict(
        completed_epochs=len(history), best_epoch=saved['epoch'],
        best_source_macro_f1=saved['best_source_macro_f1'],
        stopped_early=stale >= cfg['patience']), indent=2))
    return model, metrics, pd.DataFrame(history)


In [ ]:
# Record your hypothesis before starting the new study; do not use target outcomes.
HYPOTHESIS = (
    'Stronger source alignment may reduce source-domain separability but can remove '
    'class information; Sketch performance need not improve monotonically.')
def record_study_plan():
    destination = RESULTS / 'controlled_study'
    destination.mkdir(parents=True, exist_ok=True)
    plan = dict(method='DAN-DG', values=[0.1, 1.0, 10.0], main_lambda=1.0,
                main_sam_rho=0.05, seed=6304, hypothesis=HYPOTHESIS,
                selection='Mean source validation macro-F1 only')
    write_lock(destination / 'pre_analysis_plan.json', plan)


In [ ]:
# Choose one action, restart the kernel, and run all cells for each action.
# Order: erm, dan_dg, sam, dan_dg_lambda_0p1, dan_dg_lambda_10, diagnostics.
RUN = 'erm'
record_study_plan()
if RUN == 'erm':
    summary, provenance = evaluate_erm(ROOT)
    display(summary)
elif RUN == 'diagnostics':
    # Locks the five checkpoints and computes source-only diagnostics.
    display(source_diagnostics(ROOT))
else:
    choices = {'dan_dg': ('dan_dg', 1.0), 'sam': ('sam', None),
               'dan_dg_lambda_0p1': ('dan_dg', 0.1),
               'dan_dg_lambda_10': ('dan_dg', 10.0)}
    method, weight = choices[RUN]
    cfg = json.loads((REPO / f'task3/configs/{method}.yaml').read_text())
    if weight is not None:
        cfg['lambda_dg'] = weight
    model, metrics, history = train_pacs(DATA_ROOT, SPLIT_PATH, RESULTS / RUN, cfg, method)
    summary = pd.concat([metrics[['domain','accuracy','macro_f1']], pd.DataFrame([
        dict(domain='mean_source', accuracy=metrics.accuracy.mean(), macro_f1=metrics.macro_f1.mean()),
        dict(domain='worst_source', accuracy=metrics.accuracy.min(), macro_f1=metrics.macro_f1.min())])])
    summary.to_csv(RESULTS / RUN / 'source_validation_summary.csv', index=False)
    display(summary)
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.epoch, history.classification_loss, label='Classification CE')
    for column in ['mmd_loss', 'weighted_mmd_loss', 'perturbed_classification_loss']:
        if column in history:
            axes[0].plot(history.epoch, history[column], label=column)
    axes[0].legend()
    axes[1].plot(history.epoch, history.mean_val_macro_f1, label='Mean source macro-F1')
    axes[1].legend()
    for ax in axes: ax.set_xlabel('Epoch')
    fig.tight_layout()
    fig.savefig(RESULTS / RUN / 'training_curves.png', dpi=160)
    plt.show()
